<img width="20%" alt="EarthDaily Analytics" src="https://raw.githubusercontent.com/earthdaily/Images/main/Corporate/EarthDaily.png" style="border-radius: 15%">

# EarthDaily Agriculture - Difference Map Extraction

Development notebook for the `DifferenceExtractor` class.
Tests all three processing modes: **stats**, **links**, and **file**.

The Difference Map API computes pixel-level vegetation index differences between two satellite images,
enabling change detection and temporal comparison of field conditions.

## Step 1: Initialisation

In [ ]:
# Bootstrap: ensure src/ is on sys.path for earthdaily.agriculture imports
import sys
from pathlib import Path

_src = str(Path().resolve().parent / "src")
if _src not in sys.path:
    sys.path.insert(0, _src)

from earthdaily.agriculture.notebook_setup import init
init()

In [ ]:
from earthdaily.agriculture.services.workflow_manager import WorkflowManager
import os
manager = WorkflowManager("prod", log_to_console=True, log_level="DEBUG")

## Step 2: Get entities

### Option 1 - Load entities from EarthDaily platform

In [ ]:
manager.load_seasonfields()

print("First entity loaded:")
print(manager.sfd_list[['id', 'name']].head())

### Option 2 - Load entities from file

In [ ]:
from earthdaily.agriculture.core.geometry import load_geodataframe

file_path = "inputs/agroterrint.parquet"
manager.sfd_list = load_geodataframe(file_path)
print(f"Loaded {len(manager.sfd_list)} entities from {file_path}")
print(manager.sfd_list.columns.tolist())

## Step 3: Get coverage (two image IDs per entity)

The Difference Map API requires **two** image IDs per entity (`image_id_1` and `image_id_2`).
Use CoverageExtractor to find available images, then pick two dates to compare.

In [ ]:
from earthdaily.agriculture.extractors.coverage_function import CoverageExtractor
cov_extractor = CoverageExtractor(manager.bearer_token, manager.token_expiration, config=manager.config)

cov_extractor.setup_coverage_parameters(
    vegetation_index='NDVI',
    start_date='2025-01-01',
    end_date='2025-08-01',
    clear_cover_min=95,
)

In [ ]:
# Test coverage on a single entity
test_entity = {
    "id": "test_001",
    "geometry": "POLYGON ((-97.70066562 37.14062335, -97.69927729 37.14227539, -97.69935777 37.14233954, -97.70004188 37.14248389, -97.70008212 37.14359058, -97.69969534 37.14450741, -97.70262024 37.14450741, -97.70254625 37.14062335, -97.70066562 37.14062335))"
}

cov_result = cov_extractor.get_satellite_coverage_by_geometry(test_entity)
cov_df = cov_extractor.format_coverage_json(cov_result)
print(f"Found {len(cov_df)} images")

# Pick two images for difference comparison (first and last available)
test_entity["image_id_1"] = cov_df["image_id"].iloc[0]
test_entity["image_id_2"] = cov_df["image_id"].iloc[-1]
print(f"Image 1: {test_entity['image_id_1']}")
print(f"Image 2: {test_entity['image_id_2']}")

## Step 4: Extract difference maps

Test DifferenceExtractor with all three processing modes.

### Configure extraction (stats mode)

In [ ]:
from earthdaily.agriculture.extractors.difference_functions import DifferenceExtractor
diff_extractor = DifferenceExtractor(manager.bearer_token, manager.token_expiration, config=manager.config)

diff_extractor.setup_difference_parameters(
    product='NDVI',
    postprocess='stats',
    output_epsg=4326,
)

### Test API call

In [ ]:
print("--- Test: get_difference_map_api ---")
try:
    raw_response = diff_extractor.get_difference_map_api(test_entity)
    print("Raw API response received")
    print(type(raw_response))
except Exception as e:
    print(f"Error: {e}")

### Test safe API call

In [ ]:
print("--- Test: get_difference_map_api_safe ---")
safe_result = diff_extractor.get_difference_map_api_safe(test_entity)
print(f"Success: {safe_result['success']}")
print(f"Error: {safe_result['error']}")

### Test format_difference_stats_json

In [ ]:
print("--- Test: format_difference_stats_json ---")
if safe_result['success'] and safe_result['data']:
    stats_df = diff_extractor.format_difference_stats_json(safe_result['data'])
    print(f"Stats DataFrame: {stats_df.shape}")
    display(stats_df)
else:
    print("No data to format")

### Test process_single_entity_difference (stats)

In [ ]:
import pandas as pd

row = pd.Series({
    "id": "test_001",
    "name": "Test_Field",
    "geometry": test_entity["geometry"],
    "image_id_1": test_entity["image_id_1"],
    "image_id_2": test_entity["image_id_2"],
})

result = diff_extractor.process_single_entity_difference(row)
print(f"Error: {result['error']}")
if result['data'] is not None:
    print(f"Data shape: {result['data'].shape}")
    display(result['data'])

### Test links mode

In [ ]:
diff_extractor.setup_difference_parameters(
    product='NDVI',
    postprocess='links',
    directLinks=True,
    output_epsg=4326,
)

result_links = diff_extractor.process_single_entity_difference(row)
print(f"Error: {result_links['error']}")
if result_links['data'] is not None:
    print(f"Data shape: {result_links['data'].shape}")
    display(result_links['data'])

### Test file mode (TIFF.ZIP)

In [ ]:
diff_extractor.setup_difference_parameters(
    product='NDVI',
    postprocess='file',
    map_format='tiff.zip',
    output_path=manager.output_result_dir,
    output_epsg=4326,
)

result_file = diff_extractor.process_single_entity_difference(row)
print(f"Error: {result_file['error']}")
if result_file['data'] is not None:
    print(f"Data shape: {result_file['data'].shape}")
    display(result_file['data'])

### Visualize difference TIFF with percentile stretch

In [ ]:
import rasterio
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm

# Get the saved TIFF path from the file result
if result_file["data"] is not None and "saved_files" in result_file["data"].columns:
    tif_path = eval(result_file["data"]["saved_files"].iloc[0])[0]
    print(f"Reading: {tif_path}")

    with rasterio.open(tif_path) as src:
        data = src.read(1).astype(float)
        nodata = src.nodata
        bounds = src.bounds

        # Mask nodata
        if nodata is not None:
            data[data == nodata] = np.nan

        valid = data[~np.isnan(data)]
        print(f"Shape: {data.shape}")
        print(f"Value range: [{valid.min():.6f}, {valid.max():.6f}]")
        print(f"Mean: {valid.mean():.6f}, Std: {valid.std():.6f}")

        # Percentile stretch (2nd-98th) for robust visualization
        vmin = np.nanpercentile(data, 2)
        vmax = np.nanpercentile(data, 98)

        # Diverging colormap centered on 0 (green=increase, red=decrease)
        # If data range doesn't cross 0, fall back to simple stretch
        if vmin < 0 < vmax:
            norm = TwoSlopeNorm(vmin=vmin, vcenter=0, vmax=vmax)
        else:
            norm = None

        fig, axes = plt.subplots(1, 2, figsize=(16, 6))

        # Left: Difference map
        im = axes[0].imshow(
            data,
            cmap="RdYlGn",
            norm=norm,
            vmin=vmin if norm is None else None,
            vmax=vmax if norm is None else None,
            interpolation="nearest"
        )
        axes[0].set_title(f"Difference Map (2nd-98th percentile stretch)")
        plt.colorbar(im, ax=axes[0], label="Difference value", shrink=0.8)
        axes[0].axis("off")

        # Right: Histogram of values
        axes[1].hist(valid, bins=50, color="steelblue", edgecolor="white", alpha=0.8)
        axes[1].axvline(0, color="red", linestyle="--", linewidth=1, label="zero")
        axes[1].axvline(valid.mean(), color="orange", linestyle="-", linewidth=1, label=f"mean={valid.mean():.4f}")
        axes[1].set_title("Value Distribution")
        axes[1].set_xlabel("Difference value")
        axes[1].set_ylabel("Pixel count")
        axes[1].legend()

        plt.tight_layout()
        plt.show()
else:
    print("No TIFF file to visualize")

## Step 5: Bulk extraction

Run difference maps on multiple entities in parallel.
Each entity needs two image IDs — we pick the earliest and latest available images.

In [ ]:
# First get coverage for all entities to obtain image_ids
cov_results = cov_extractor.process_entity_coverage_bulk_parallel(
    entity_list=manager.sfd_list.head(10),
    max_workers=5,
    skip_export=True,
)

cov_df = cov_results["results_df"]
print(f"Coverage results: {len(cov_df)} rows")

# Aggregate: pick first and last image per entity for difference comparison
image_pairs = (
    cov_df.groupby("id")["image_id"]
    .agg(image_id_1='first', image_id_2='last')
    .reset_index()
)

# Merge image pairs back onto original entity data
entities = manager.sfd_list.head(10).copy()
diff_input = entities.merge(
    image_pairs,
    on="id",
    how="inner"
)

print(f"Difference input: {len(diff_input)} entities with image pairs")
display(diff_input[["id", "image_id_1", "image_id_2"]].head())

In [ ]:
# Setup difference in stats mode
diff_extractor.setup_difference_parameters(
    product='NDVI',
    postprocess='stats',
    output_epsg=4326,
)

diff_results = diff_extractor.process_entity_difference_bulk_parallel(
    entity_list=diff_input,
    max_workers=5,
    output_path=manager.output_result_dir,
    skip_export=False,
    prefix='difference',
)

print(f"Total: {diff_results['total_calculations']}")
print(f"Successful: {diff_results['successful_calculations']}")
print(f"Failed: {diff_results['failed_calculations']}")
if not diff_results['results_df'].empty:
    display(diff_results['results_df'].head())

## Step 6: Workflow Integration Test

Test the DifferenceExtractor in the pattern used by `WorkflowManager.run_workflow()`.

**YAML config reference:**
```yaml
steps:
  - name: difference
    depends_on: coverage
    extractor: DifferenceExtractor
    module: earthdaily.agriculture.extractors.difference_functions
    setup: { method: setup_difference_parameters, params: { product: NDVI, postprocess: stats } }
    run: { method: process_entity_difference_bulk_parallel }
```

In [ ]:
# Workflow-style dynamic import & setup (mirrors WorkflowManager._execute_single_step)
import importlib

module_path = "earthdaily.agriculture.extractors.difference_functions"
class_name = "DifferenceExtractor"

mod = importlib.import_module(module_path)
ExtractorClass = getattr(mod, class_name)

wf_diff = ExtractorClass(manager.bearer_token, manager.token_expiration, config=manager.config)

# Setup with workflow parameters
wf_diff.setup_difference_parameters(
    product="NDVI",
    postprocess="stats",
)

# --- Coverage step: get image IDs for the test entity ---
from earthdaily.agriculture.extractors.coverage_function import CoverageExtractor

wf_cov = CoverageExtractor(manager.bearer_token, manager.token_expiration, config=manager.config)
wf_cov.setup_coverage_parameters(
    vegetation_index="NDVI",
    start_date="2025-01-01",
    end_date="2025-08-01",
    clear_cover_min=95,
)

wf_test_entity = {
    "id": manager.sfd_list.iloc[0]["id"],
    "geometry": manager.sfd_list.iloc[0]["geometry"],
}

wf_cov_result = wf_cov.get_satellite_coverage_by_geometry(wf_test_entity)
wf_cov_df = wf_cov.format_coverage_json(wf_cov_result)
print(f"Found {len(wf_cov_df)} images for workflow test entity")

print("Workflow-style DifferenceExtractor ready")

### Test with workflow entity format

In [ ]:
import pandas as pd

# Build test entity with image_id_1 and image_id_2 from workflow coverage results
test_row = pd.Series({
    "id": manager.sfd_list.iloc[0]["id"],
    "geometry": manager.sfd_list.iloc[0]["geometry"],
    "image_id_1": wf_cov_df["image_id"].iloc[0],
    "image_id_2": wf_cov_df["image_id"].iloc[-1],
})
print(f"Entity: {test_row['id']}")
print(f"image_id_1: {test_row['image_id_1']}")
print(f"image_id_2: {test_row['image_id_2']}")

# Process single entity
result = wf_diff.process_single_entity_difference(test_row)

if result["data"] is not None:
    print(f"Result columns: {list(result['data'].columns)}")
    display(result["data"].head())
else:
    print(f"Error: {result['error']}")